Youtube video clip of the senate hearing mentioning 85% of child deaths due to influenza were children who did not receive a flu vaccine: https://www.youtube.com/watch?v=ukp8tlzzBKI. Senator Michael Bennet questioned RFK Jr. about changes to vaccine recommendations, pointing out that 2025 saw the highest number of childhood (pediatric) flu deaths in modern American history. 

Primary Questions: 
1. Is there a corellation between pediatric deaths due to flu and flu vaccine coverage? 
2. What about RFK Jr.'s claim that flu vaccines are so innefective that they are not worth recommending? 

Criteria:
1. Your project has a recognizable and reproducible “data analytics workflow.” [Example: First the data is acquired, then necessary transformations and clean-up are performed, then the analysis and presentation work is performed]
2. Project includes data from at least two different types of data sources (e.g., two or more of these: (1) relational, (2) scraped web page, (3) web API.)
3. Project includes at least one data transformation operation. [Examples: transforming from wide to long; converting columns to date format]
4. Project includes at least one grouping or aggregation.
5. Project includes at least one statistical analysis and at least one graphics that describes or validates your data.
6. Project includes at least one graphic that supports your conclusion(s).
7. Project includes at least one statistical analysis that supports your conclusion(s).
8. Project includes at least one feature that we did not cover in class! There are many examples: “I used Bokeh; I created a decision tree; I ranked the results; I created my presentation slides directly from my Jupyter Notebook; I figured out to use OAuth 2.0…”
9. Presentation. Was the presentation delivered in the allotted time (3 to 5 minutes)?
10. Presentation. Did you show (at least) one challenge you encountered in code and/or data, and what you did when you encountered that challenge? If you didn’t encounter any challenges, your assignment was clearly too easy for you!
11. Presentation. Did the audience come away with a clear understanding of your motivation for undertaking the project?
12. Presentation. Did the audience come away with a clear understanding of at least one insight you gained or conclusion you reached or hypothesis you “confirmed” (rejected or failed to reject…)?
13. Code and data. Have you delivered the submitted code and data where it is reproducible and self-contained—preferably in a Jupyter Notebook on GitHub? Am I able to fully reproduce your results with what you’ve delivered? You won’t receive full credit if your code references data on your local machine!
14. Code and data. Does all of the delivered code run without errors?
15. Deadline management. Were your draft project proposal, project, and presentation delivered on time? Any part of the project that is turned in late will receive a maximum grade of 80%. Please turn in your work on time! You are of course welcome to deliver ahead of schedule!


Datasets used:
CDC Influenza-Associated Pediatric Mortality: https://gis.cdc.gov/GRASP/Fluview/PedFluDeath.html (csv)
CDC Data API: [data.cdc.gov](https://data.cdc.gov/)  (API)
CDC FluView Website: https://www.cdc.gov/fluview/index.html (scraped weekly data)
CDC Estimated US Flu Disease Burden: https://www.cdc.gov/flu-burden/php/data-vis/index.html (csv)



Import data from the U.S. Centers for Disease Control (CDC) data API, and load CDC Flu disease burden data, scraped pediatric mortality data, and flu related pediatric mortality data. 

In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

# dependencies for getting CDC api data
from sodapy import Socrata

# dependencies for web scraping
from bs4 import BeautifulSoup
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# dependencies for data visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

load_dotenv(override=True) 
MY_APP_TOKEN = os.getenv("API_KEY")

CDC_DATASET_ID = "vh55-3he6"

# Use the provided app token to bypass basic throttling
client = Socrata("data.cdc.gov", MY_APP_TOKEN)

# note 2009 did have mixed data for the Influenza A (H1N1) 2009 Monovalent vaccine
query = """
SELECT 
    *
WHERE 
    geography = 'United States'
    AND vaccine = 'Seasonal Influenza'
    AND dimension_type = 'Age'
    AND dimension = '6 Months - 17 Years'
LIMIT 50000
"""

try:
    results = client.get(CDC_DATASET_ID, query=query)
    cdc_api_df = pd.DataFrame.from_records(results)
           
    print(f"Success! Retrieved {len(cdc_api_df)} records.")
    
    
except Exception as e:
    print(f"Query Failed: {e}")


# load local test data from previously scraped data (scraping data was taking roughly 20 min each attempt)
test_scraped_df = pd.read_csv("flu_scraped_df.csv")

# load csv download data for disease burden and pediatric death
disease_burden_data_df = pd.read_csv("flu-disease-burden-past-season-rounded-estimates-all-ages.csv")
ped_death_df = pd.read_csv("Weekly_Pediatric_Deaths.csv")




Success! Retrieved 173 records.


This solution for scraping flu data from the CDC website uses selenium to launch a background browser to render the webpages since the CDC FluView site uses client side rendering to populate the data on the page. Once the page has been loaded and populated in the background browser, then BeautifulSoup is used to scrape the weekly hospitalization rate and pediatric deaths. 

Note 1: This is not an optimal solution since the data is also included in a downloadable csv, this is designed to demonstrate data gathering using web scraping. FluView dashboards start in October 2024. Also 2026-week-13 on the CDC site has an issue retrieving data from its backend, so unfortunately that data is not available. There are multiple issues with trying to scrape long-term data on the FluView website. First, many of the older reports have been converted to a PDF which would have added an additional layer to the scraping process. Also, for some reason there is a fair amount of inconsistency in the format of the page and the text associated with the hospitalization rate and weekly pediatric deaths. 

Note 2: The scraping process takes 15-20 minutes to complete. A csv of the resulting file (flu_scraped_df.csv) is included just in case things need to be sped up.

FluView Homepage: https://www.cdc.gov/fluview/index.html 
Disease burden data: https://www.cdc.gov/flu-burden/php/data-vis/index.html

In [ ]:
def scrape_fluview_structural(start_year, start_week, end_year, end_week):
    
    all_data = []

    # setup background browser
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    print("Launching background browser...")
    driver = webdriver.Chrome(options=chrome_options)

    for year in range(start_year, end_year + 1):
        for week in range(1, 54):

            # Only pull data for weeks between the start year/week and end year/week
            if year == start_year and week < start_week:
                continue
            if year == end_year and week > end_week:
                break

            week_str = f"{week:02d}"
            url = f"https://www.cdc.gov/fluview/surveillance/{year}-week-{week_str}.html"

            try:
                print(f"Loading: {year} Week {week_str}...")
                hosp_rate = None
                pediatric_deaths = None

                # launch page in background browser
                driver.get(url)
                time.sleep(10)  # need a buffer for the client side data to load

                soup = BeautifulSoup(driver.page_source, "html.parser")

                # parse the layout by div sections
                cards = soup.find_all(
                    "div", class_="cove-visualization__inner"
                )

                if not cards:
                    print(f"-> No report elements found for {year}-Wk{week_str}")
                    continue

                # drill through the sections and find data
                for card in cards:
                    # Isolate card header text
                    header_tag = card.find(["h2", "header"])
                    header_text = header_tag.get_text(strip=True).lower() if header_tag else ""

                    # Isolate the text block inside this specific card
                    content_block = card.find("div", class_="cove-prose")
                    if not content_block:
                        continue

                    card_text = content_block.get_text(separator=" ", strip=True).replace(
                        "\xa0", " "
                    )

                    # Get Hospitalization Rate
                    # Matches ANY variant: "weekly hospitalization rate" OR "cumulative hospitalization rate"
                    if "hospitalization rate" in card_text.lower():
                        # Captures the first digit/decimal group (e.g., 0.4 or 12.5)
                        rate_match = re.search(r"([\d.]+)", card_text)
                        if rate_match:
                            hosp_rate = float(rate_match.group(1))

                    # Get Pediatric Deaths
                    if "pediatric deaths" in header_text:
                        # Find the span used for the data header
                        large_num_span = card.find("span", style=re.compile(r"size:\s*1\.5rem"))

                        if large_num_span and large_num_span.get_text(strip=True):
                            num_text = large_num_span.get_text(strip=True)
                            death_match = re.search(r"([\d,]+)", num_text)
                            if death_match:
                                pediatric_deaths = int(death_match.group(1).replace(",", ""))
                        else:
                            # Fallback: Enforce that the number must precede 'influenza-associated'
                            strict_match = re.search(r"([\d,]+)\s+influenza-associated", card_text)
                            if strict_match:
                                pediatric_deaths = int(strict_match.group(1).replace(",", ""))
                            else:
                                pediatric_deaths = 0


                all_data.append(
                    {
                        "year": year,
                        "week": week,
                        "hospitalization_rate": hosp_rate,
                        "pediatric_deaths": pediatric_deaths,
                    }
                )

                print(
                    f"   Success -> Rate: {hosp_rate} | Deaths: {pediatric_deaths}"
                )

            except Exception as e:
                print(f"   Error parsing {year}-Wk{week_str}: {e}")
                continue

    driver.quit()

    # extract to a pandas dataframe
    df = pd.DataFrame(
        all_data,
        columns=["year", "week", "hospitalization_rate", "pediatric_deaths"],
    )
    return df


# Start week is set to 2024/36 because prior reports have been converted to PDF
if __name__ == "__main__":
    flu_scraped_df = scrape_fluview_structural(
        start_year=2024, start_week=36, end_year=2026, end_week=17
    )

    print("\nExtraction finished. Final Pandas Dataframe view:")
    print(flu_scraped_df.head())
    flu_scraped_df.to_csv('flu_scraped_df.csv', index=False)

Launching background browser...
Loading: 2024 Week 36...
   Success -> Rate: 0.1 | Deaths: 3
Loading: 2024 Week 37...
   Success -> Rate: 0.0 | Deaths: 0
Loading: 2024 Week 38...
   Success -> Rate: 0.0 | Deaths: 1
Loading: 2024 Week 39...
   Success -> Rate: 0.1 | Deaths: 1
Loading: 2024 Week 40...
   Success -> Rate: 0.0 | Deaths: 0
Loading: 2024 Week 41...
   Success -> Rate: 0.1 | Deaths: 0
Loading: 2024 Week 42...
   Success -> Rate: 0.1 | Deaths: 0
Loading: 2024 Week 43...
   Success -> Rate: 0.4 | Deaths: 0
Loading: 2024 Week 44...
   Success -> Rate: 0.7 | Deaths: 1
Loading: 2024 Week 45...
   Success -> Rate: 0.9 | Deaths: 0
Loading: 2024 Week 46...
   Success -> Rate: 1.2 | Deaths: 1
Loading: 2024 Week 47...
   Success -> Rate: 1.6 | Deaths: 0
Loading: 2024 Week 48...
   Success -> Rate: 2.1 | Deaths: 0
Loading: 2024 Week 49...
   Success -> Rate: 3.1 | Deaths: 0
Loading: 2024 Week 50...
   Success -> Rate: 4.8 | Deaths: 2
Loading: 2024 Week 51...
   Success -> Rate: 7.7 | De

Clean up data and transform as needed. Mostly creating a common datetime format to make merging data easier.

Note: June is often omitted from each year since the CDC considers a flu season to be from July to May of the following year. 


In [ ]:
def clean_api_data(cdc_api_df):
    
    cdc_api_df = cdc_api_df.copy()
    numeric_cols = ['coverage_estimate', 'population_sample_size', 'month']

    # Split the seasons by calendar year since the flu season starts in July and convert to datetime
    first_year = cdc_api_df['year_season'].str.split('-').str[0].astype(int)
    cal_year = np.where(cdc_api_df['month'].astype(int) > 6, first_year, first_year + 1)
    cdc_api_df['date'] = pd.to_datetime(
        pd.Series(cal_year, index=cdc_api_df.index).astype(str) + '-' + cdc_api_df['month'].astype(str)
    )

    # sort by year/month
    cdc_api_df[numeric_cols] = cdc_api_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    return cdc_api_df.sort_values(by='date').reset_index(drop=True)


def clean_disease_burden_data(disease_burden_data_df):
    disease_burden_data_df = disease_burden_data_df.copy()
    # format the year to match other datasets
    disease_burden_data_df['Flu Season'] = disease_burden_data_df['Flu Season'].str.split('-').str[0]
    return disease_burden_data_df


# clean up scraped data
def clean_scraped_data(scraped_df):
    scraped_df = scraped_df.copy()
    # convert year/week into a datetime objects
    scraped_df['date'] = pd.to_datetime(
        scraped_df['year'].astype(str) + '-' + scraped_df['week'].astype(str) + '-1', 
        format='%G-%V-%u'
    )

    scraped_df['month'] = scraped_df['date'].dt.strftime('%Y-%m')

    # Transform any null values then group by month and aggregate and sort
    scraped_df['pediatric_deaths'] = scraped_df.groupby('month')['pediatric_deaths'].transform(lambda x: x.fillna(x.mean()))
    consolidated_df = scraped_df.groupby('month').agg(
        hospitalization_rate=('hospitalization_rate', 'mean'),
        pediatric_deaths=('pediatric_deaths', 'sum')
    ).reset_index()

    consolidated_df = consolidated_df.sort_values('month')
    return consolidated_df

# clean up pediatric death data
def clean_ped_death_data(ped_df):
    ped_df = ped_df.copy()
    
    columns_to_remove = ['Season', 'PREVIOUS WEEK DEATHS', 'CURRENT WEEK DEATHS']
    ped_df = ped_df.drop(columns=columns_to_remove, errors='ignore')
    
    # convert week/year data into datetime objects
    if 'Week Number' in ped_df.columns:
        # Extract components into temporary variables instead of adding them to ped_df
        year_part = ped_df['Week Number'].str.split('-').str[0]
        week_part = ped_df['Week Number'].str.split('-').str[1].str.zfill(2)
        
        # Convert directly into a datetime object
        ped_df['date'] = pd.to_datetime(
            year_part + '-W' + week_part + '-1', 
            format='%Y-W%W-%w', 
            errors='coerce'
        )
        
        # Remove 'Week Number' 
        ped_df = ped_df.drop(columns=['Week Number'])
        
    return ped_df

cleaned_cdc_api_df = clean_api_data(cdc_api_df)
cleaned_test_scraped_df = clean_scraped_data(test_scraped_df)
cleaned_disease_burden_data_df = clean_disease_burden_data(disease_burden_data_df)
cleaned_ped_death_df = clean_ped_death_data(ped_death_df)

# create excel spreadsheets to look over data
cleaned_cdc_api_df.to_excel('cdc_api_df_cleaned.xlsx', index=False)
cleaned_test_scraped_df.to_excel('consolidated_scraped_data_df_cleaned.xlsx', index=False)
cleaned_disease_burden_data_df.to_excel('disease_burden_data_df_cleaned.xlsx', index=False)
cleaned_ped_death_df.to_excel('ped_death_df_cleaned.xlsx', index=False)

Visualization and analysis of the correlation between flu related pediatric deaths, and estimated flu vaccine coverage:

1. The time period referenced in the Congressional hearing was the 2025 flu season which runs from July 2024 - June 2025. 
2. Each flu season sees a peak in pediatric deaths as the vaccine coverage rises, and then the peak drops off as vaccine coverage reaches roughly 50% of the pediatric population. During the 2024-2025 flu season, the maximum vaccine coverage peaks around 50%.
3. 2024-2025 flu season has the lowest pediatric peak vaccination coverage recorded in the past 15 years. 
4. 2020-2022 are anomalies in the pediatric death data primarily because of precautions taken as a result of the Covid-19 pandemic. 
5. Data for June is omitted in the CDC data as a gap between the flu seasons. It is not statistically significant since the flu season peaks in the fall and winter. 

Note: According to the National Institute of Health (NIH) the H3N2 strain of influenza has a higher hospitalization and mortality rates and is responsible for the recent spikes in fatalities seen in the 2014–2015, 2017–2018, 2022–2023 flu seasons. The predominant strains of influenza in the 2024-2025 flu season are A(H1N1)pdm09 and A(H3N2) - H1N1 being the strain responsible for the 1918 influenza pandemic. 
Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC7144439/



In [ ]:

# chart Pediatric Deaths and Estimated Flu Vaccine Coverage from July 2010 - July 2025
def chart_coverage_and_deaths(cdc_api_df, ped_df):
    cdc_api_df = cdc_api_df.copy()
    ped_df = ped_df.copy()

    # Set date columns to datetime objects
    cdc_api_df['date'] = pd.to_datetime(cdc_api_df['date'])
    ped_df['date'] = pd.to_datetime(ped_df['date'])

    # Aggregate weekly pediatric deaths dataset by month
    ped_monthly = ped_df.groupby(ped_df['date'].dt.to_period('M'))['Pediatric Deaths'].sum().reset_index()
    ped_monthly['date'] = ped_monthly['date'].dt.to_timestamp()

    # Set start and end month
    all_months = pd.date_range(start='2010-07-01', end='2025-07-01', freq='MS')
    df_all = pd.DataFrame({'date': all_months})

    # Merge the data on date
    df_all = df_all.merge(cdc_api_df[['date', 'coverage_estimate']], on='date', how='left', validate='one_to_one')
    df_all = df_all.merge(ped_monthly, on='date', how='left', validate='one_to_one')

    # Forward fill missing entries since you cannot "unvaccinate" people
    # eg. If you have 65% coverage in March, you will have at least 65% coverage in April
    df_all = df_all.ffill()

    # Create a dual-axis line chart
    fig, ax1 = plt.subplots(figsize=(14, 7))

    # Coverage Estimate Chart
    color = 'tab:blue'
    ax1.set_xlabel('Year', fontsize=12, labelpad=10)
    ax1.set_ylabel('Coverage Estimate (%)', color=color, fontsize=12)
    line1 = ax1.plot(df_all['date'], df_all['coverage_estimate'], color=color, linewidth=2, label='Coverage Estimate (%)')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, linestyle='--', alpha=0.5)

    # Total Monthly Pediatric Deaths Chart
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Pediatric Deaths (Monthly Sum)', color=color, fontsize=12)
    line2 = ax2.plot(df_all['date'], df_all['Pediatric Deaths'], color=color, linewidth=2, linestyle='--', label='Pediatric Deaths')
    ax2.tick_params(axis='y', labelcolor=color)

    # Combine legends for coverage and deaths
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')

    # Format and save
    plt.title('Influenza Vaccine Coverage Estimate and Pediatric Deaths Over Time (July 2010 - July 2025)', fontsize=14, pad=15)
    ax1.xaxis.set_major_locator(mdates.YearLocator())
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig('coverage_vs_deaths_validated.png', dpi=300)
    plt.close()

chart_coverage_and_deaths(cleaned_cdc_api_df, cleaned_ped_death_df)


Analysis of flu vaccination coverage vs flu related hospitalization rates:

1. There seems to be little correlation between the peak vaccination coverage and the number of hospitalizations for each flu season. 
2. At face value this seems to support Secretary Robert F. Kennedy Jr.'s point that flu vaccines are relatively ineffective. However when taking into account vaccination coverage vs the number of pediatric deaths, it appears that even if a child is hospitalized due to flu, they are less likely to die from flu related complications if they are vaccinated. 



In [ ]:
def chart_flu_season_metrics(api_df, burden_df):
    
    api_df = api_df.copy()
    burden_df = burden_df.copy()
    
    # format flu season data so they can be merged and get the peak coverage estimate per flu season
    burden_df['Flu Season'] = burden_df['Flu Season'].astype(str).str.split('-').str[0].astype(int)
    api_df['season_start_year'] = api_df['year_season'].astype(str).str.split('-').str[0].astype(int)
    peak_coverage = api_df.groupby('season_start_year')['coverage_estimate'].max().reset_index()
    
    
    
    # Merge datasets on the starting year of the flu season
    merged_data = pd.merge(
        burden_df, 
        peak_coverage, 
        left_on='Flu Season', 
        right_on='season_start_year', 
        how='inner',
        validate='one_to_one'
    ).sort_values('Flu Season')
    
    # Set up chart
    fig, ax1 = plt.subplots(figsize=(13, 6.5))
    
    # Format chart
    season_labels = [f"{year}-{str(year+1)[2:]}" for year in merged_data['Flu Season']]
    x_positions = np.arange(len(season_labels))
    
    # Set up Hospitalizations bar chart
    color_bar = 'tab:blue'
    ax1.set_xlabel('Flu Season', fontsize=12, labelpad=12)
    ax1.set_ylabel('Hospitalizations', color=color_bar, fontsize=12)
    ax1.bar(
        x_positions, 
        merged_data['Hospitalizations'], 
        color=color_bar, 
        alpha=0.6, 
        width=0.45, 
        label='Hospitalizations'
    )
    ax1.tick_params(axis='y', labelcolor=color_bar)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, pos: format(int(val), ',')))
    ax1.grid(True, linestyle='--', alpha=0.3, axis='y')
    
    # Set up Peak Vaccination Coverage Estimate
    ax2 = ax1.twinx()
    color_line = 'tab:red'
    ax2.set_ylabel('Peak Vaccination Coverage Estimate (%)', color=color_line, fontsize=12)
    ax2.plot(
        x_positions, 
        merged_data['coverage_estimate'], 
        color=color_line, 
        marker='o', 
        linewidth=2.5, 
        label='Peak Coverage Estimate (%)'
    )
    ax2.tick_params(axis='y', labelcolor=color_line)
    ax2.set_ylim(0, 110)
    
    # add percentage labels to each peak coverage estimate value
    for x, y in zip(x_positions, merged_data['coverage_estimate']):
        ax2.annotate(
            f"{y:.1f}%", 
            (x, y), 
            textcoords="offset points", 
            xytext=(0, 10),
            ha='center', 
            fontsize=9, 
            color=color_line, 
            weight='bold'
        )


    # Set up labels and legend
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(season_labels, rotation=45)
    plt.title('Peak Vaccination Coverage and Total Hospitalizations by Flu Season', fontsize=14, pad=15)
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left')
    
    # Save chart
    plt.tight_layout()
    plt.savefig('vax_coverage_hospitalizations.png', dpi=300)
    plt.close()

chart_flu_season_metrics(cleaned_cdc_api_df, cleaned_disease_burden_data_df)

Analysis of preliminary findings for the 2025-2026 Flu Season by making a comparison of 2025 and projected 2026 flu seasons.

Background: The peak vaccination coverage estimate for the 2024-2025 flu season was 50.2%, and the projected coverage for 2025-2026 is 49.3% according to a preliminary report released by the CDC on May 9, 2026. Source: https://www.cdc.gov/respiratory-viruses/data/vaccination-trends.html

Conclusions based on the data:
1. Even though vaccination rates are slightly lower in the 2025-2026 flu season, there will be fewer pediatric deaths than the previous season. 
2. The data indicates that the vaccine alone is not the only factor that impacts flu related pediatric mortality.



In [15]:
def chart_yoy_deaths(scraped_df):
    
    scraped_df = scraped_df.copy()

    scraped_df['date'] = pd.to_datetime(scraped_df['month'])
    scraped_df['year'] = scraped_df['date'].dt.year
    scraped_df['month_num'] = scraped_df['date'].dt.month

    # Group data by flu season
    scraped_df['season'] = scraped_df.apply(
        lambda row: f"{row['year']}-{str(row['year']+1)[2:]}" if row['month_num'] >= 7 
        else f"{row['year']-1}-{str(row['year'])[2:]}", axis=1
    )

    # Set order of months for flu season
    month_order = [7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]
    month_labels = ['Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
    target_seasons = ['2024-25', '2025-26']

    # Initialize matrix with placeholders then populate with date/season info
    yoy_deaths = {s: {m: 0 for m in month_order} for s in target_seasons}

    for _, row in scraped_df.iterrows():
        s = row['season']
        m = row['month_num']
        if s in yoy_deaths:
            yoy_deaths[s][m] = row['pediatric_deaths']

    # Create chart
    fig, ax = plt.subplots(figsize=(11, 6.5))
    x_positions = np.arange(len(month_order))

    # Format chart
    for s in target_seasons:
        deaths_series = [yoy_deaths[s][m] for m in month_order]
        
        if s == '2025-26':
            ax.plot(x_positions, deaths_series, color='red', linewidth=3.5, label=f"Season {s}")
        else:
            ax.plot(x_positions, deaths_series, linestyle='--', linewidth=1.5, label=f"Season {s}")

    ax.set_ylabel('Pediatric Deaths', fontsize=12)
    ax.set_xlabel('Month of Flu Season', fontsize=12, labelpad=10)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(month_labels)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    ax.set_title('Year-over-Year Comparison: Pediatric Deaths by Flu Season', fontsize=14, pad=15)

    # Add comparative vaccination coverage for 2024-2025 vs 2025-2026
    text_str = (
        "The peak flu vaccination coverage estimate for the 2024-2025 flu season was 50.2%.\n"
        "As of May 9, 2026 it is projected that roughly 49.3% of the pediatric population will be vaccinated."
    )
    plt.subplots_adjust(bottom=0.22)

    fig.text(
        0.5, 0.04, text_str, ha='center', fontsize=11, 
        bbox=dict(boxstyle='round,pad=0.5', facecolor='whitesmoke', edgecolor='lightgray', alpha=0.8)
    )

    # Save chart
    plt.savefig('yoy_deaths_two_seasons.png', dpi=300)
    plt.close()

chart_yoy_deaths(cleaned_test_scraped_df)


